# Bases: dissociação e pOH
### Química — funções inorgânicas

**Objetivo:** Classificar bases de Arrhenius e relacionar concentração de OH⁻, pOH e pH.

### Como usar este notebook
Execute as células em ordem. As de código conferem as contas e geram as visualizações. No 3D, **arraste com o mouse para girar em 360°** e use a roda para zoom.
A primeira célula instala somente dependências ausentes. O 3D precisa de acesso a `cdn.jsdelivr.net` no navegador e de um notebook confiável (trusted); se a rede bloquear o visualizador, o desenho 2D continua disponível.

Material para estudo teórico: não prove, toque nem misture produtos químicos para identificar uma função química.

In [ ]:
# Preparação idempotente: usa o Python do próprio kernel.
import importlib.util
import subprocess
import sys

for pacote, modulo in [("rdkit", "rdkit"), ("py3Dmol", "py3Dmol"),
                       ("matplotlib", "matplotlib")]:
    if importlib.util.find_spec(modulo) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pacote], check=True)

import math
import matplotlib.pyplot as plt
import py3Dmol
from rdkit import Chem
from rdkit.Chem import rdDepictor
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import SVG, display

## 1. Conceito principal

Bases de **Arrhenius** liberam $\mathrm{OH^-}$ por dissociação em água. Bases de **Brønsted–Lowry** aceitam prótons; essa definição também inclui a amônia, que não possui OH na fórmula.

$$\mathrm{NaOH(s)\rightarrow Na^+(aq)+OH^-(aq)}$$
$$\boxed{\mathrm{pOH}\approx-\log_{10}([OH^-]/(1\ \mathrm{mol/L})),\qquad pH+pOH=14{,}00}$$

Usaremos soluções ideais diluídas a **25 °C**; em outras temperaturas, o valor de pKᵥ muda.

| Fórmula | Nome | Classificação |
|---|---|---|
| NaOH | hidróxido de sódio | monobase, forte, solúvel |
| Ca(OH)₂ | hidróxido de cálcio | dibase, forte na fração dissolvida, pouco solúvel |
| Fe(OH)₃ | hidróxido de ferro(III) | tribase, muito pouco solúvel |

Conte os grupos OH para classificar hidróxidos. **Solubilidade e força não são sinônimos.** Hidróxidos iônicos não são moléculas isoladas.

## 2. Exemplo resolvido passo a passo

Para NaOH completamente dissociado a $0{,}0050\ \mathrm{mol/L}$:

$$[OH^-]=1\cdot c=5{,}0\times10^{-3}\ \mathrm{mol/L}$$
$$\mathrm{pOH}=-\log_{10}(5{,}0\times10^{-3})\approx2{,}30$$
$$\mathrm{pH}=14{,}00-2{,}30=\boxed{11{,}70}$$

## 3. Conferindo o cálculo com Python

In [ ]:
c = 0.0050  # mol/L
poh = -math.log10(c)
ph = 14.0 - poh
print(f"[OH-] = {c:.4f} mol/L; pOH = {poh:.2f}; pH = {ph:.2f}")
assert math.isclose(ph, 11.698970004336019)

## 4. Visualização

### 4.1 Modelo 3D interativo

**Íon hidróxido (OH⁻).** O = vermelho, H = branco; ligação ilustrativa de 0,97 Å. Mostramos o íon poliatômico, não uma suposta molécula de NaOH. A carga negativa aparece no desenho 2D. A hidratação e o contraíon Na⁺ não estão representados.

In [ ]:
# Coordenadas didáticas em Å; não são uma simulação de solução.
SMILES = '[OH-]'
COORDENADAS = [(0.0, 0.0, 0.0), (0.97, 0.0, 0.0)]
mol = Chem.MolFromSmiles(SMILES)
if mol is None:
    raise ValueError(f"SMILES inválido: {SMILES!r}")
mol = Chem.AddHs(mol)
if mol.GetNumAtoms() != len(COORDENADAS):
    raise ValueError("O número de coordenadas não corresponde ao de átomos.")
conf = Chem.Conformer(mol.GetNumAtoms())
for indice, posicao in enumerate(COORDENADAS):
    conf.SetAtomPosition(indice, posicao)
mol.AddConformer(conf)
visor = py3Dmol.view(width=640, height=400)
visor.addModel(Chem.MolToMolBlock(mol), "mol")
visor.setStyle({"stick": {"radius": 0.13, "colorscheme": "Jmol"},
                 "sphere": {"scale": 0.28, "colorscheme": "Jmol"}})
visor.setBackgroundColor("#ffffff")
visor.zoomTo()
visor.show()

### 4.2 Representação 2D

Cada traço representa uma ligação; dois traços representam uma ligação dupla.

In [ ]:
# Estrutura 2D: ligações covalentes com traços; íons separados por ponto
# no SMILES permanecem separados, sem inventar uma ligação covalente.
plano = Chem.Mol(mol)
rdDepictor.Compute2DCoords(plano)
desenho = rdMolDraw2D.MolDraw2DSVG(520, 300)
desenho.drawOptions().bondLineWidth = 2.2
desenho.DrawMolecule(plano)
desenho.FinishDrawing()
display(SVG(desenho.GetDrawingText()))

### 4.3 Gráfico de dados

In [ ]:
cs = [10**(-k / 20) for k in range(40, 101)]
plt.plot(cs, [14 + math.log10(c) for c in cs])
plt.xscale("log")
plt.xlabel("Concentração de NaOH (mol/L)")
plt.ylabel("pH (25 °C, aproximação ideal)")
plt.title("Base forte: concentração e pH")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Como interpretar o resultado

O traço O–H é interno ao íon OH⁻. Não representa uma ligação covalente Na–O. Ao diluir a base dez vezes, o pOH aumenta uma unidade e o pH diminui uma unidade neste regime.

## 6. Exercício de fixação

Considere Ca(OH)₂ já dissolvido a 0,0010 mol/L, completamente dissociado. Qual é [OH⁻] e qual é o pH a 25 °C? Lembre-se dos dois grupos OH.

Tente resolver antes de executar a resposta abaixo.

In [ ]:
c_ex = 0.0010  # mol/L de Ca(OH)2 dissolvido
oh_ex = 2 * c_ex
ph_ex = 14 + math.log10(oh_ex)
print(f"[OH-] = {oh_ex:.4f} mol/L; pH = {ph_ex:.2f}")
assert math.isclose(ph_ex, 11.301029995663981)

## 7. Resumo

- Hidróxidos liberam OH⁻ em água.
- Nome: hidróxido de + elemento, com NOX quando necessário.
- Cada mol de Ca(OH)₂ dissolvido fornece dois mol de OH⁻.
- pH + pOH = 14,00 a 25 °C no modelo adotado.

## Referências

- IUPAC. *Compendium of Chemical Terminology (Gold Book)*: https://goldbook.iupac.org/
- ATKINS, P.; JONES, L. *Princípios de Química*. Bookman.